## Employee Attrition Prediction (Open Ended Lab)

Goal: Predict employee attrition (`Attrition`) and identify key factors that contribute to employees leaving the company.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
)

pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

## 1) Load Dataset

In [ ]:
import os
from pathlib import Path

dataset_name = "WA_Fn-UseC_-HR-Employee-Attrition.csv"

candidate_paths = [
    Path.cwd() / dataset_name,
    Path.cwd() / "lab_07_oel" / dataset_name,
    Path(__file__).resolve().parent / dataset_name if "__file__" in globals() else None,
    Path.home() / "Downloads" / dataset_name,
    Path.home() / "Downloads" / "archive (1)" / dataset_name,
    Path(r"c:\Users\PMLS\Downloads\archive (1)\WA_Fn-UseC_-HR-Employee-Attrition.csv"),
]

# Remove None entries and pick the first existing path.
candidate_paths = [p for p in candidate_paths if p is not None]
csv_path = next((p for p in candidate_paths if p.exists()), None)

if csv_path is None:
    # Final fallback: recursive search from current working directory.
    recursive_matches = list(Path.cwd().rglob(dataset_name))
    if recursive_matches:
        csv_path = recursive_matches[0]

if csv_path is None:
    print("Current working directory:", Path.cwd())
    print("Checked paths:")
    for p in candidate_paths:
        print(" -", p)
    raise FileNotFoundError(
        f"{dataset_name} not found. Place it in the notebook folder or Downloads/archive (1)."
    )

df = pd.read_csv(csv_path)
print(f"Loaded from: {csv_path}")
print("Shape:", df.shape)
display(df.head())

FileNotFoundError: Dataset not found. Put WA_Fn-UseC_-HR-Employee-Attrition.csv in lab_07_oel or update candidate_paths.

## 2) Quick EDA

In [3]:
print("Column types:")
display(df.dtypes)

print("\nTop missing values:")
display(df.isna().sum().sort_values(ascending=False).head(15))

print("\nAttrition distribution (count):")
display(df["Attrition"].value_counts())

print("\nAttrition distribution (proportion):")
display(df["Attrition"].value_counts(normalize=True).rename("proportion"))

Column types:


NameError: name 'df' is not defined

## 3) Preprocessing and Split

In [ ]:
drop_cols = ["EmployeeNumber", "EmployeeCount", "Over18", "StandardHours"]
work_df = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

work_df["Attrition"] = work_df["Attrition"].map({"Yes": 1, "No": 0})

X = work_df.drop(columns=["Attrition"])
y = work_df["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Numeric features:", len(num_cols), "| Categorical features:", len(cat_cols))

## 4) Train Models

In [ ]:
logreg_model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
])

rf_model = Pipeline([
    ("prep", preprocessor),
    (
        "clf",
        RandomForestClassifier(
            n_estimators=400,
            random_state=42,
            class_weight="balanced_subsample",
        ),
    ),
])

logreg_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

print("Both models trained successfully.")

## 5) Evaluate Models

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    }

    print(f"\n===== {name} =====")
    print(pd.Series(metrics))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    return metrics, y_prob

log_metrics, log_prob = evaluate_model("Logistic Regression", logreg_model, X_test, y_test)
rf_metrics, rf_prob = evaluate_model("Random Forest", rf_model, X_test, y_test)

results = pd.DataFrame([log_metrics, rf_metrics]).set_index("Model")
display(results.sort_values("ROC-AUC", ascending=False))

## 6) ROC Curves

In [ ]:
plt.figure(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, log_prob, name="Logistic Regression")
RocCurveDisplay.from_predictions(y_test, rf_prob, name="Random Forest")
plt.title("ROC Curves")
plt.grid(alpha=0.3)
plt.show()

## 7) Feature Importance (Random Forest)

In [ ]:
rf_clf = rf_model.named_steps["clf"]
ohe = rf_model.named_steps["prep"].named_transformers_["cat"].named_steps["onehot"]

feature_names = num_cols + list(ohe.get_feature_names_out(cat_cols))
importances = rf_clf.feature_importances_

imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False)

display(imp_df.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(data=imp_df.head(12), x="importance", y="feature", hue="feature", dodge=False, legend=False)
plt.title("Top 12 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 8) Conclusion Template

- Better model based on ROC-AUC/F1: **fill after running**
- Top attrition drivers (from importance table): **fill top 5-10**
- Suggested HR actions:
  - Control excessive overtime
  - Improve work-life balance policies
  - Target retention plans for higher-risk roles and tenure groups